In [ ]:
# import sys
# import os

# Add the LayerIF_Pruning_New directory to sys.path
# sys.path.append(os.path.abspath("../LayerIF_Pruning_New"))


# To run main.py as a script
!CUDA_VISIBLE_DEVICES=0,1,2,3 python  ../LayerIF_Pruning_New/main.py \
    --model mistralai/Mistral-7B-v0.1 \
    --cache_dir llm_weights/ \
    --prune_method wanda_ww \
    --sparsity_ratio 0.5 \
    --save results/mistral-IF-smoothed-0.5 \
    --ww_metric IF-300-96-smoothed \
    --ww_metric_cache ../LayerIF_Pruning_New/data/mistral-7b \
    --epsilon 0.3 \
    --eval_wikitext False \
    --eval_zero_shot 

In [ ]:
# change the directory path again to mdl/
sys.path.insert(0, os.path.abspath("."))
import pruning
import util

model_metadata = '../Expert_Allocation/layerIF_outputs/mistral_mola_46810_224_glue_cola_all/mola_lora_summary.json'
# retained_base_params = calculate_retained_metrics(model_metadata, sparsity_ratio=0.3)

connections_per_layer = util.get_all_layer_connections(model_metadata)
print(f"connection_per_layer: {connections_per_layer}")
sparsity_target = sum(connections_per_layer) * 0.4
lambda_value, rho_l = pruning.prune(n_l=connections_per_layer,
                                    b=16,
                                    eta=0.1, 
                                    layer_qualities=util.get_IF(), 
                                    k=1, 
                                    sparsity=sparsity_target, 
                                    epsilon=0.2)

rho_l_str = " ".join(str(round(value, 2)) for value in rho_l)


In [ ]:
rho_l_str

In [ ]:
!CUDA_VISIBLE_DEVICES=0,1,2,3 python  main.py \
    --model mistralai/Mistral-7B-v0.1 \
    --cache_dir llm_weights/ \
    --prune_method wanda_ww \
    --sparsity_ratio 0.5 \
    --save results/mistral-IF-smoothed-0.5-mdl \
    --ww_metric IF-300-96-smoothed \
    --ww_metric_cache ./data/mistral-7b/ \
    --epsilon 0.3 \
    --eval_wikitext False \
    --eval_zero_shot \
    --rho_l {rho_l_str}